# 记忆压缩

上一章，智能体有了记忆，但也留下一个麻烦：聊得越久，每次要发送的聊天记录就越长，越来越贵、越来越慢，最后还会超出模型的上下文窗口。

想想我们自己是怎么记东西的。上了一学期的课，没有人能逐字记住老师说过的每一句话。我们会**做笔记**：把讲过的内容浓缩成几条要点，最近讲的内容则记得比较清楚。

学习助手也可以这样做：

```
压缩前：[第 1 轮问答] [第 2 轮问答] [第 3 轮问答] ... [最近一轮问答]
                     ↓ 把较早的内容写成摘要
压缩后：[摘要：小明正在学这本书，问过……]        [最近一轮问答]
```

- **较早的对话**，交给大模型写成一段简短的摘要；
- **最近的几条消息**，原样保留。

这样，记忆的长度就不会无限增长了。而写摘要这件事，正好可以交给大模型自己来做。

In [ ]:
import os
from abc import ABC, abstractmethod
from enum import Enum
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

## 一个小工具：note

从这一章开始，智能体在回答之前，会在“幕后”做一些事情，比如压缩记忆，后面还会调用工具、查资料。为了把这些幕后的步骤和问答区分开，我们写一个小函数 `note`：它把信息缩进两格，前面加一个 `*`，多行的内容也会对齐。

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

## 会做笔记的 Memory

`Memory` 类在上一章的基础上，多了做笔记的本领。先看它的几个参数：

- `provider`：写摘要也要调用大模型，所以 Memory 需要知道用哪个模型。
- `max_messages`：消息超过多少条就压缩。
- `max_tokens`：内容超过多长就压缩。
- `keep_messages`：压缩时，最近的几条消息原样保留，不写进摘要。

为了在演示中马上看到效果，这里的数值设得很小：超过 3 条消息就压缩，只保留最近 1 条。真实的应用里，通常会设成几十条消息、几千个 token。

再看几个方法：

- `add`：每加一条消息，就检查一下是否超过了限制，超过了就调用 `summarize`。
- `summarize`：做笔记的核心，分三步：
  1. 把消息分成两部分：较早的 `to_summarize`，和最近的 `to_keep`。
  2. 把较早的消息拼成一段文字，加上“请把下面的对话压缩成一段简短的摘要”，交给大模型。这里用了一条专门的系统消息 `SUMMARIZE_SYSTEM_MSG`，让大模型扮演“摘要助手”。
  3. 用摘要替换掉较早的消息：新的记忆 = 一条摘要 + 最近的消息。
- `token_count`：估算记忆有多长。准确计算 token 需要用到模型的分词器，这里简单地数字符个数，作为粗略的估计。

为什么把摘要放进一条 `UserMessage`，还要加上前缀 `Historical Summary:`？因为对话最好从用户的消息开始、一问一答地进行，有的 API 对此有严格的要求。把摘要当作一条用户消息放在最前面，格式最稳妥；前缀则告诉大模型：这不是小明刚说的话，而是之前对话的摘要。

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1):
        self.provider = provider
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, message: Message):
        self.messages.append(message)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

## Agent 的变化

`Agent` 只改了一个地方：创建记忆时，要把 `provider` 传给 `Memory`，因为写摘要也要用到大模型。

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage):
        self.provider = provider
        self.system = system
        self.memory = Memory(provider)

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            user_message = UserMessage(message)
            response = self.provider.call(system or self.system, self.memory.messages + [user_message])
            self.memory.add(user_message)
            self.memory.add(response)
            print(f"AI: {response.summary()}")

## 压缩以后还记得吗

这次问三个问题：先自我介绍，再问一个别的问题，最后看看学习助手还记不记得小明的名字。

运行时注意看缩进的 `* summary` 那几行：那就是记忆被压缩后的样子。

In [ ]:
provider = OpenAIProvider()
agent = Agent(provider, system_message)
agent.chat(["你好，我叫小明，正在学这本书。", "第 1 部分讲了什么？", "我叫什么名字？"])  # still remembered after summarizing?
# or call agent.chat() without questions to type your own

看看压缩以后，记忆里还剩下什么：

In [ ]:
print(agent.memory.history())

最早的几条消息已经不见了，取而代之的是一段摘要。只要摘要里提到了小明的名字，学习助手就仍然能答出来。

## 做笔记的代价

压缩记忆解决了“越聊越长”的问题，但也有代价：

- **会丢失细节**。摘要是浓缩过的，没写进摘要的内容就真的忘了。比如小明一开始说过“我喜欢简短的解释”，如果摘要里没有这一句，学习助手之后就不会记得。
- **要多花一次调用**。写摘要也要调用大模型，也要花钱。

除了写摘要，还有一种更简单的做法：**只保留最近的 N 条消息**，更早的直接丢掉，叫作“滑动窗口”。它不用额外调用大模型，但丢得更彻底。而对于真正重要、需要长期记住的信息（比如小明的学习进度），第 16 章会介绍另一种办法：长期记忆。

## 动手试试

- 把 `max_messages` 改成 6、`keep_messages` 改成 2，多聊几轮，看看压缩发生的时机有什么变化。
- 修改 `SUMMARIZE_USER_MSG`，要求摘要里“必须保留用户的名字、学习进度和偏好”，看看摘要有什么不同。
- 试着实现滑动窗口：在 `summarize` 里不调用大模型，直接 `self.messages = to_keep`。比较一下两种做法的效果。

## 小结

- 记忆太长时，可以像做笔记一样：把较早的对话交给大模型写成**摘要**，最近的几条消息原样保留。
- `Memory` 通过 `max_messages` 和 `max_tokens` 判断什么时候压缩，用 `keep_messages` 决定保留多少条原始消息。
- 摘要以一条带前缀的 `UserMessage` 放在记忆最前面，既保证了格式，也让大模型知道这是摘要。
- 压缩会丢失细节，也会多花一次调用。另一种简单的做法是滑动窗口。

现在学习助手能聊得很久了，但程序一关，所有记忆就都没了。下一章，我们把记忆**保存到文件里**，让小明下次打开时可以接着聊。